# Fine-Tuning DistilBERT on Parent-Teacher Communication Dataset


### This code block imports all the essential libraries needed to build and fine-tune a DistilBERT text classification model. It begins by importing **PyTorch (`torch`)**, which handles the deep learning computations and GPU acceleration, followed by **Pandas (`pd`)** for loading and cleaning datasets. The **`train_test_split`** function from Scikit-learn is used to divide the data into training and testing sets to ensure fair model evaluation, while **`Dataset`** from the Hugging Face library converts the data into a compatible format for transformer models. The **`DistilBertForSequenceClassification`** and **`DistilBertTokenizerFast`** classes load the pre-trained DistilBERT model for text classification and handle text preprocessing, respectively. The **`TrainingArguments`** and **`Trainer`** modules simplify model training by managing hyperparameters, evaluation, and checkpointing automatically. **NumPy (`np`)** supports efficient numerical operations, while **Scikit-learn’s metrics** such as accuracy, precision, recall, and F1-score help measure model performance. Finally, the **`pipeline`** function from Transformers provides an easy interface for performing predictions or inference using the trained model.


In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import pipeline

## **1. Setup and Data Preparation.**

- This code block begins by importing the dataset using the Pandas library, where the file Test.csv is read and stored in a DataFrame named df. Once loaded, a confirmation message is printed, and the first few rows of the dataset are displayed to give an overview of its structure. To ensure data consistency, only the necessary columns, message_text and sentiment, are retained, while any rows containing missing values are removed. Since machine learning models require numeric labels, the sentiment column is converted to integers—mapping positive to 1 and negative to 0—if it originally contains text values. The dataset is then split into training and evaluation subsets using an 80:20 ratio with the train_test_split function, ensuring that the model’s performance can be validated on unseen data. Finally, both subsets are converted into Hugging Face Dataset objects, making them compatible with the Transformer library for model training and evaluation.

- Device setup computes device for training is configured using PyTorch. The code automatically detects whether a GPU (Graphics Processing Unit) is available through CUDA; if it is, the GPU is selected to significantly accelerate model training. Otherwise, the model defaults to using the CPU. The setup also includes a print statement that confirms which device is being used. If a GPU is available, it displays the GPU’s name. This setup ensures that the model runs efficiently and utilizes available computational resources to their fullest.

In [ ]:
# 1. LOAD DATA
df = pd.read_csv('/content/school_reports_messages_augmented_tone_harder_longer.csv')
print("Dataset loaded successfully!")
print(df.head())

# Ensure the dataset has the right columns
df = df[['message_text', 'sentiment']].dropna()

# Convert text labels (if not numeric) to integers
# Example: Positive = 1, Negative = 0
if df['sentiment'].dtype == 'object':
    df['sentiment'] = df['sentiment'].map({'neutral': 1, 'negative': 0}).fillna(0).astype(int)

# Split into train and eval
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

# 2. DEVICE SETUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available, using CPU.")

Dataset loaded successfully!
                                        message_text sentiment        tone
0  according daughter group students bullied son ...  negative  not_urgent
1  child experienced kids making impolite jokes s...  negative  not_urgent
2  someone informed classroom environment hot col...   neutral      urgent
3  son said school facilities need cleaning kids ...   neutral  not_urgent
4  students mentioned school facilities need clea...   neutral  not_urgent
Using GPU: Tesla T4


***TEST SET ***

In [ ]:
eval_df.to_csv('eval_set.csv', index=False)
print("Test set saved to 'eval_set.csv'. You can download it from the Colab file browser.")

Test set saved to 'eval_set.csv'. You can download it from the Colab file browser.


## **2. Load pre-trained model definition and tokenization.**

- This code block handles the transformation of raw text into a numerical form that the DistilBERT model can process. The model name distilbert/distilbert-base-uncased is specified, and its corresponding tokenizer is loaded using DistilBertTokenizerFast.from_pretrained(). A tokenization function is defined to preprocess the message_text column by applying truncation and padding, ensuring all text sequences have consistent lengths. This function is then applied to both the training and evaluation datasets using the .map() method for batch tokenization. Afterward, the sentiment column is renamed to labels, as required by the Hugging Face Trainer for supervised learning tasks. Both tokenized datasets are then formatted as PyTorch tensors using .set_format(), which prepares them for efficient model input during training and evaluation.

- In Model Definition stage, the pre-trained DistilBERT model for sequence classification is loaded and customized for the sentiment analysis task. The model is initialized using DistilBertForSequenceClassification.from_pretrained(), with the number of output labels determined by the unique values in the dataset’s sentiment column (e.g., two for positive and negative). The model is then transferred to the chosen computing device—GPU if available or CPU otherwise—for optimized performance. A confirmation message prints the model name to indicate that it has been successfully loaded. This setup allows the DistilBERT model to leverage its pre-trained language understanding capabilities while being fine-tuned on the specific dataset.

- The Metrics Function defines a custom function named compute_metrics() to evaluate the model’s performance after each epoch. The function uses np.argmax() to determine predicted classes by selecting the label with the highest probability. It then calculates four essential performance metrics using Scikit-learn functions: accuracy, which measures the overall correctness of predictions; F1-score, which provides a balanced measure of precision and recall; precision, which assesses how many predicted positives are correct; and recall, which measures how many actual positives were successfully detected by the model. The function returns these metrics in a dictionary format, allowing the Hugging Face Trainer to automatically compute and display them during training and evaluation. This ensures that the model’s learning progress can be monitored effectively, providing a clear understanding of its predictive performance.

In [ ]:
# 3. TOKENIZATION
MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["message_text"], truncation=True, padding=True)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Rename label column for Trainer compatibility
tokenized_train = tokenized_train.rename_column("sentiment", "labels")
tokenized_eval = tokenized_eval.rename_column("sentiment", "labels")

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 4. MODEL DEFINITION
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(df['sentiment'].unique())).to(device)
print(f"Model loaded: {MODEL_NAME}")

# 5. METRICS FUNCTION
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    precision = precision_score(p.label_ids, preds, average="weighted", zero_division=0)
    recall = recall_score(p.label_ids, preds, average="weighted")
    f1 = f1_score(p.label_ids, preds, average="weighted")
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

Map:   0%|          | 0/2849 [00:00<?, ? examples/s]

Map:   0%|          | 0/713 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: distilbert/distilbert-base-uncased


## **3. Define training configuration and initialize trainer**

- In Training Setup, the training configuration for the DistilBERT model is defined using the TrainingArguments class, which sets all the parameters that control the model’s learning process. The output_dir="./results_experiment" specifies where the trained model and logs will be saved, while num_train_epochs=5 means the model will train for five complete passes through the dataset to improve performance. The learning_rate=5e-5 controls how much the model’s weights are adjusted during optimization, ensuring stable and gradual learning. Both per_device_train_batch_size=8 and per_device_eval_batch_size=8 determine how many samples are processed at once during training and evaluation, balancing memory efficiency with model accuracy. The weight_decay=0.05 adds regularization to prevent overfitting, and warmup_ratio=0.1 gradually increases the learning rate at the start of training for smoother convergence. The lr_scheduler_type="cosine_with_restarts" allows the learning rate to oscillate and recover from performance plateaus. Additionally, gradient_accumulation_steps=2 accumulates gradients over multiple batches to simulate a larger batch size, while max_grad_norm=0.8 clips gradients to avoid instability during updates. The logging_dir="./logs" and logging_steps=20 ensure progress and metrics are recorded at regular intervals. The model evaluates and saves checkpoints after every epoch (eval_strategy="epoch" and save_strategy="epoch") to monitor improvement, while load_best_model_at_end=True ensures the final model used is the best one based on validation results. The best model is selected using the F1-score (metric_for_best_model="f1", greater_is_better=True), prioritizing balanced precision and recall. Mixed precision training (fp16=torch.cuda.is_available()) is enabled if GPU hardware supports it, improving speed and memory efficiency. The report_to="none" disables unnecessary reporting tools, and seed=42 ensures consistent and reproducible results. Finally, the Trainer class initializes the training process by combining the model, training arguments, tokenized datasets, metrics computation, and tokenizer, automating the entire fine-tuning pipeline for DistilBERT and ensuring a structured, efficient, and reproducible sentiment analysis workflow.

In [ ]:
# 6. TRAINING SETUP

training_args = TrainingArguments(
    output_dir="./results_experiment",
    num_train_epochs=3,               # Reduced epochs from 5 to 4
    learning_rate=3e-5,               # Lower LR from 3e-5 to 2e-5
    per_device_train_batch_size=16,  # Reduced batch size from 32 to 16
    per_device_eval_batch_size=16,
    weight_decay=0.4,               # Reduced weight decay from 0.1 to 0.04
    warmup_ratio=0.10,               # Reduced warmup ratio from 0.8 to 0.15
    lr_scheduler_type="cosine",
    gradient_accumulation_steps=1,
    max_grad_norm=0.05,               # Increased max_grad_norm from 0.05 to 1.0
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

/tmp/ipython-input-1535300904.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
eval_df.to_csv('eval_set.csv', index=False)
print("Test set saved to 'eval_set.csv'. You can download it from the Colab file browser.")

## **4. Finetune and Save the Model.**

- In this part of the code, the fine-tuning and evaluation stages of the DistilBERT model are executed. The line print("\n--- Starting Fine-Tuning ---") displays a message indicating that the training process is about to begin. The command trainer.train() initiates the actual fine-tuning process using the Hugging Face Trainer class, which handles all the training operations such as feeding batches of tokenized data into the model, performing forward and backward propagation, updating model weights, and monitoring progress across epochs. This process allows the model to learn from the labeled training data and adjust its parameters to minimize prediction errors. After training is completed, print("\n--- Evaluation Results ---") signals the start of the evaluation phase. The line results = trainer.evaluate() then assesses the trained model’s performance using the validation dataset, computing metrics such as accuracy, precision, recall, and F1-score as defined in the earlier compute_metrics() function. Finally, print(results) displays the evaluation results, allowing researchers to analyze how effectively the model performs on unseen data. Together, these steps ensure that the model is both trained and validated properly, providing measurable insights into its accuracy and overall reliability for sentiment analysis tasks.

In [ ]:
# 7. TRAINING
print("\n--- Starting Fine-Tuning ---")
trainer.train()

# 8. EVALUATION
print("\n--- Evaluation Results ---")
results = trainer.evaluate()
print(results)


--- Starting Fine-Tuning ---


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.056800,1.809712,0.798036,0.799183,0.798036,0.797981
2,0.092300,1.955590,0.796634,0.798111,0.796634,0.796542
3,0.019800,1.998960,0.803647,0.804256,0.803647,0.803641



--- Evaluation Results ---


{'eval_loss': 1.8097124099731445, 'eval_accuracy': 0.7980364656381487, 'eval_precision': 0.7991827097932374, 'eval_recall': 0.7980364656381487, 'eval_f1': 0.7979808414352634, 'eval_runtime': 0.5229, 'eval_samples_per_second': 1363.557, 'eval_steps_per_second': 86.059, 'epoch': 3.0}


## **5. Inference Process.**

- This section performs inference, where the trained DistilBERT model is used to predict the sentiment of new messages. A sentiment analysis pipeline is created using the fine-tuned model and tokenizer, with the device automatically set to GPU if available or CPU otherwise. A list of sample texts is provided to test how well the model can identify positive, negative, or urgent tones in unseen data. The loop runs each message through the sentiment analyzer, retrieving the predicted label and confidence score for each. Finally, the program prints the results, showing how accurately the model interprets different message sentiments based on its training.

In [ ]:
sentiment_analyzer = pipeline(
    'sentiment-analysis',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

urgency_analyzer = pipeline(
    'sentiment-analysis',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

print("Sentiment and Urgency analysis pipelines initialized.")
print("\n--- Enter Your Text (type 'exit' to quit) ---")

while True:
    user_input_text = input("Please enter a message: ")

    if user_input_text.lower() == 'exit':
        print("Exiting combined analysis.")
        break
    elif user_input_text:
        # Sentiment Analysis
        result_sentiment = sentiment_analyzer(user_input_text)[0]
        label_sentiment = result_sentiment['label']
        score_sentiment = result_sentiment['score']

        sentiment_mapping = {
            'LABEL_0': 'negative',
            'LABEL_1': 'neutral'
        }
        mapped_sentiment = sentiment_mapping.get(label_sentiment, label_sentiment)

        # Urgency Analysis
        result_urgency = urgency_analyzer(user_input_text)[0]
        label_urgency = result_urgency['label']
        score_urgency = result_urgency['score']

        urgency_mapping = {
            'LABEL_0': 'not_urgent',
            'LABEL_1': 'urgent'
        }
        mapped_urgency = urgency_mapping.get(label_urgency, label_urgency)

        print(f"\nText: {user_input_text}\nSentiment: {mapped_sentiment} (Score: {score_sentiment:.4f})\nUrgency: {mapped_urgency} (Score: {score_urgency:.4f})\n")
    else:
        print("No text entered. Please enter a message or 'exit' to quit.")

Device set to use cuda:0
Device set to use cuda:0


Sentiment and Urgency analysis pipelines initialized.

--- Enter Your Text (type 'exit' to quit) ---
Please enter a message: students mentioned student mocked daughter front others restroom smells unpleasvnt The increased academic pressure due to curriculum changes has led to heightened stress and anxiety among students, prompting administration to evaluate the effectiveness of existing counseling services.	

Text: students mentioned student mocked daughter front others restroom smells unpleasvnt The increased academic pressure due to curriculum changes has led to heightened stress and anxiety among students, prompting administration to evaluate the effectiveness of existing counseling services.	
Sentiment: negative (Score: 0.9971)
Urgency: not_urgent (Score: 0.9971)

Please enter a message: students mentioned student pushed daughter today student threatened hurt The increased academic pressure due to curriculum changes has led to heightened stress and anxiety among students, prompting